# 07 张量并行与序列并行

## 简介

本笔记本介绍 Megatron-LM 风格的张量并行（Tensor Parallelism, TP）和序列并行（Sequence Parallelism, SP）原理。
TP 将权重矩阵切分到多张 GPU 上并行计算，减少单卡显存并加速矩阵运算。
SP 在 TP 区域内沿序列维度切分 LayerNorm/Dropout 的激活值，进一步减少激活显存占用。

In [ ]:
import torch
import torch.distributed as dist
import sys; sys.path.insert(0, '..')
from parallel.communication.setup import get_rank, get_world_size
from parallel.tensor_parallel.column_parallel import column_parallel_linear, split_weight_column
from parallel.tensor_parallel.row_parallel import row_parallel_linear, split_weight_row
from parallel.tensor_parallel.sequence_parallel import scatter_along_seq, gather_along_seq
from parallel.tensor_parallel.megatron_style import megatron_transformer_block_fwd

print(f"当前 rank: {get_rank()}")
print(f"world_size: {get_world_size()}")
print(f"dist 已初始化: {dist.is_initialized()}")

## 1. 列并行（Column Parallel）

列并行将权重矩阵按**列**方向切分。每个 GPU 持有 `W` 的一部分列（`W[:, local_cols]`），
输入 `x` 在所有 rank 上相同。

```
  W_full 形状: (dim, hidden_dim)
  ┌──────────────────────────────┐
  │  Rank 0  │ Rank 1 │ Rank 2  │  ... 按列切分
  └──────────────────────────────┘

  每个 rank 计算: y_local = x @ W_local  (shape: B, S, hidden_local)
  通信操作: All-Gather 拼接各 rank 的局部输出 → 完整 y (B, S, hidden_dim)
```

**通信复杂度**: 每层 1 次 All-Gather，总通信量 ≈ N（N = 激活值大小 × world_size）

In [ ]:
# 列并行概念演示：权重按列切分
dim, hidden_dim = 64, 256
world_size = 4

W_full = torch.randn(dim, hidden_dim)

print("列并行 (Column Parallel) 权重切分:")
print(f"  完整权重形状: {list(W_full.shape)}")
print(f"  切分方式: 沿 dim=1 (列方向) 均分为 {world_size} 份")
print(f"  每份形状: ({dim}, {hidden_dim // world_size})")

for rank in range(world_size):
    chunk_size = hidden_dim // world_size
    w_local = W_full[:, rank * chunk_size : (rank + 1) * chunk_size]
    print(f"  Rank {rank}: 持有 W[:, {rank*chunk_size}:{(rank+1)*chunk_size}] -> 形状 {list(w_local.shape)}")

print(f"\n输入 x 在所有 rank 上相同: (batch=2, seq=4, dim={dim})")
print(f"各 rank 输出: (2, 4, {hidden_dim // world_size}) -> All-Gather 后: (2, 4, {hidden_dim})")

# 验证: 完整计算 vs 切分后计算
x = torch.randn(2, 4, dim)
y_full = x @ W_full
print(f"\n完整计算输出形状: {list(y_full.shape)}")
print(f"列并行 + All-Gather 输出形状: {list(y_full.shape)} (理论上等价)")

## 2. 行并行（Row Parallel）

行并行将权重按**行**方向切分。输入也相应切分，输出的规约方式与列并行不同。

```
  W_full 形状: (hidden_dim, dim)
  ┌─── Rank 0 ───┐
  ├─── Rank 1 ───┤  按行切分
  ├─── Rank 2 ───┤
  └─── Rank 3 ───┘

  每个 rank 计算: y_local = x_local @ W_local  (shape: B, S, dim)
  通信操作: All-Reduce (SUM) 求和所有 rank 的局部输出
```

**Megatron TP 核心模式**: 列并行 → 行并行，中间无需通信！
这是因为列并行的输出恰好已经按列切分，可以直接作为行并行的输入。

In [ ]:
# 行并行概念演示：权重按行切分
hidden_dim, dim = 256, 64
world_size = 4

W_full = torch.randn(hidden_dim, dim)

print("行并行 (Row Parallel) 权重切分:")
print(f"  完整权重形状: {list(W_full.shape)}")
print(f"  切分方式: 沿 dim=0 (行方向) 均分为 {world_size} 份")

for rank in range(world_size):
    chunk_size = hidden_dim // world_size
    w_local = W_full[rank * chunk_size : (rank + 1) * chunk_size]
    print(f"  Rank {rank}: 持有 W[{rank*chunk_size}:{(rank+1)*chunk_size}] -> 形状 {list(w_local.shape)}")

print(f"\n输入也需要按列切分: x_local 形状为 (2, 4, {hidden_dim // world_size})")
print(f"输出: All-Reduce (SUM) 后每个 rank 得到 (2, 4, {dim})")

# === Megatron 列→行 Pipeline ===
print("\n" + "=" * 50)
print("Megatron TP 核心模式: 列并行 → 行并行 (中间无需 all-gather!)")
print("=" * 50)

B, S, D_full = 2, 4, 256
D_intermediate = 512  # 中间隐藏维度

print(f"""
  输入 x:            ({B}, {S}, {D_full})    ← 所有 rank 相同
       │
       ▼ Column Parallel (W1: {D_full} x {D_intermediate})
  列输出 y_col:      ({B}, {S}, {D_intermediate // world_size})  ← 已隐式切分!
       │
       ▼ 无需通信! y_col 直接作为行并行输入
       │
       ▼ Row Parallel (W2: {D_intermediate // world_size} x {D_full})
  行输出 y_row:      ({B}, {S}, {D_full})     ← All-Reduce 求和
""")

print("★ 列并行的输出天然按列切分 → 直接喂给行并行 → 省去 1 次 all-gather!")
print("★ 仅需在行并行输出端做 1 次 all-reduce → 通信量减半")

## 3. 序列并行（Sequence Parallel）

序列并行在 TP 区域内沿 **序列维度** 切分 LayerNorm / Dropout 的激活值，进一步减少激活显存。

**核心思想**: LayerNorm 和 Dropout 是逐 token 独立运算的，不需要跨 token 交互，
因此可以将不同 segment 的 token 分布到不同 GPU 上并行计算。

```
完整激活: (B=2, S=16, D=256)
  ┌──────────────────────┐
  │ Rank0: tokens[0:4]   │
  ├──────────────────────┤
  │ Rank1: tokens[4:8]   │
  ├──────────────────────┤
  │ Rank2: tokens[8:12]  │
  ├──────────────────────┤
  │ Rank3: tokens[12:16] │
  └──────────────────────┘

显存节省: ≈ 1 / tp_size (激活值部分)
代价: 进出 SP 区域需要 scatter/gather 通信
```

In [ ]:
# 序列并行概念演示
B, S, D = 2, 16, 256
world_size = 4

x = torch.randn(B, S, D)

print("序列并行 (Sequence Parallel) 切分:")
print(f"  完整激活形状: {list(x.shape)}")
print(f"  切分方式: 沿 dim=1 (seq_len) 均分为 {world_size} 份")
print(f"  每份形状: (2, {S // world_size}, 256)")

for rank in range(world_size):
    chunk_size = S // world_size
    x_local = x[:, rank * chunk_size : (rank + 1) * chunk_size]
    print(f"  Rank {rank}: tokens[{rank*chunk_size}:{(rank+1)*chunk_size}] -> 形状 {list(x_local.shape)}")

# 显存节省计算
bytes_per_element = 2  # fp16
total_activation_mb = B * S * D * bytes_per_element / (1024 ** 2)
per_rank_sp_mb = B * (S // world_size) * D * bytes_per_element / (1024 ** 2)
print(f"\n激活显存分析 (fp16):")
print(f"  无 SP: 每卡 {total_activation_mb:.2f} MB")
print(f"  有 SP: 每卡 {per_rank_sp_mb:.2f} MB")
print(f"  节省: {(1 - 1/world_size)*100:.0f}% (激活值部分)")

# SP 通信开销
print(f"\nSP 通信模式:")
print(f"  进入 SP 区域: 沿 seq 维度 scatter (本地切分，零通信)")
print(f"  离开 SP 区域: All-Gather 恢复完整序列 (通信量 = (P-1)/P × 激活大小)")
print(f"  总 SP 通信 ≈ 2 × All-Gather / Transformer 层")

## 4. Megatron 风格 TP+SP 完整 Transformer 块

综合列并行、行并行和序列并行的完整前向流程：

```
          输入 x (B, S, D) [完整序列]
                │
    ┌── SP 区域 ─┤
    │           ▼  Scatter: 沿 seq 切分
    │    LayerNorm / RMSNorm (各 rank 算自己的 segment)
    │           ▼  All-Gather: 恢复完整序列 (离开 SP)
    ├── TP 区域 ─┤
    │           ▼  Column Parallel: QKV 投影 (各 rank 部分列)
    │           ▼  Attention (各 rank 独立计算部分 heads)
    │           ▼  Row Parallel: Output 投影 (All-Reduce 求和)
    │           ▼  Scatter: 切分回 seq (进入 SP)
    ├── SP 区域 ─┤
    │           ▼  LayerNorm / RMSNorm
    │           ▼  All-Gather (离开 SP)
    ├── TP 区域 ─┤
    │           ▼  Column Parallel: Gate+Up 投影
    │           ▼  Activation (GELU/SiLU) 
    │           ▼  Row Parallel: Down 投影 (All-Reduce 求和)
    │           ▼  Scatter (进入 SP)
    └───────────┘
          输出 (B, S, D) [完整序列]
```

**核心优势**: f 在 TP 区域内无冗余激活存储，SP 额外减少 Norm/Dropout 的激活显存。
总激活显存减少约 1/tp_size（理论值，实际还受通信开销影响）。

In [ ]:
# Megatron 风格 TP+SP 块演示
from parallel.tensor_parallel.column_parallel import split_weight_column
from parallel.tensor_parallel.row_parallel import split_weight_row

B, S, D = 2, 8, 16  # 小尺寸演示
world_size = 2
hidden_local = D // world_size

x = torch.randn(B, S, D)

# 构造所有权重
w_qkv_full = torch.randn(D, 3 * hidden_local)  # 简化为 3*hidden_local
w_o_full = torch.randn(hidden_local, D)
w_gate_up_full = torch.randn(D, 2 * hidden_local)
w_down_full = torch.randn(hidden_local, D)

print("Megatron TP+SP Transformer 块权重形状:")
print(f"  QKV 投影 (列并行):  全量 {list(w_qkv_full.shape)}")
print(f"  Output 投影 (行并行): 全量 {list(w_o_full.shape)}")
print(f"  Gate+Up 投影 (列并行): 全量 {list(w_gate_up_full.shape)}")
print(f"  Down 投影 (行并行):   全量 {list(w_down_full.shape)}")

print(f"\nTP 通信汇总 (每 Transformer 层, world_size={world_size}):")
print(f"  1. QKV 输出端:  1 次 All-Gather (列并行)")
print(f"  2. Attention 输出端: 1 次 All-Reduce (行并行)")
print(f"  3. Gate+Up 输出端: 1 次 All-Gather (列并行)")
print(f"  4. Down 输出端:     1 次 All-Reduce (行并行)")
print(f"  SP 通信 (可选):    + 4 次 All-Gather (进出 SP)")
print(f"  总计: 2 All-Gather + 2 All-Reduce (纯 TP) 或 +4 All-Gather (TP+SP)")

# 模拟 SP 下激活显存对比
print(f"\n激活显存对比 (fp16, B=2, S=8, D=16):")
seq_chunk = S // world_size
full_act_mb = B * S * D * 2 / (1024**2)
sp_act_mb = B * seq_chunk * D * 2 / (1024**2)
print(f"  无 SP: {full_act_mb:.6f} MB / rank")
print(f"  有 SP: {sp_act_mb:.6f} MB / rank (每次 Norm 时)")
print(f"  节省比例: {(1 - sp_act_mb/full_act_mb)*100:.0f}%")

## 5. 关键要点总结

1. **列并行**: 权重按列切分 → 输入相同 → 输出需 All-Gather 拼接
2. **行并行**: 权重按行切分 → 输入对应切分 → 输出需 All-Reduce 求和
3. **Megatron 核心模式**: Column → Row，中间无需 All-Gather，通信量减半
4. **序列并行**: 在 TP 区域内沿 seq 维度切分 Norm/Dropout 激活值，减少显存约 1/tp_size
5. **SP 通信开销**: 每进出 SP 区域一次需 1 次 All-Gather，按需权衡启用
6. **TP 的局限**: 单层内通信密集（对卡间带宽要求高），通常 TP 仅在节点内（8 GPU）使用
7. **组合策略**: TP(节点内) + PP(节点间) + DP(全局) 是大模型训练的标准配置